In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_openai import OpenAIEmbeddings
# from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import ChatOllama
# from langchain_classic.chains.question_answering import load_qa_chain


C:\Users\abraa\AppData\Local\Temp\ipykernel_26336\1613335223.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# Loading AI Models (Embeddings and LLMs)
embeddings_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'} # Change to 'cuda' if you have an Nvidia GPU
)

llm = ChatOllama(model="llama3", temperature=0)

C:\PROJETOS\RAG_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2013.09it/s]


In [3]:
# Loading PDF Files...
pdf_link = "1758887557527-attachment.pdf"
loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()

In [4]:
# Chunk splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4000,
    chunk_overlap = 20,
    length_function = len,
    add_start_index = True
)

chunks = text_splitter.split_documents(pages)

In [ ]:
# Saving in Vector DB - Chroma

db = Chroma.from_documents(chunks, embedding=embeddings_model, persist_directory="text_index")

In [ ]:
# Loading db
vectorDb = Chroma(persist_directory="text_index", embedding_function=embeddings_model)

# Load retriever
retriever = vectorDb.as_retriever(search_kwargs={"k": 3})

# Chain = Construção da cadeia de prompt para chamada do LLM.

system_prompt = ("Use os seguintes pedaços de contexto para responder à pergunta. " 
                 "Se não souber a resposta, apenas diga que não sabe, e que não pode criar uma resposta sem contexto.\n\n"
                 "Contexto: \n{context}")

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

question = "Qual é o assunto principal do documento?"
answer = rag_chain.invoke({"input": question})

print("Resposta do LLM: ", answer["answer"])